## FormalVizWidget

A generic `anywidget` for animating an SVG whose visual state is driven by
a formal model. This notebook is a reusable framework for implementing animation for the spectabular specifcation libary. A
model specific example notebook is given in `elevator/Elevator.ipynb`.

To use, import the notebook with the `%run` shell magic

```
%run "../FormalVizWidget.ipynb"
```

and then supply:

- an SVG with element `id`s to animate,
- a `state` dict shape,
- `widget[key] = (fn, method)` bindings mapping state to element
  properties (see "Binding SVG properties to state" below),
- a Spectabular relation spec, as a class variable on a `FormalVizWidget`
  subclass.

In [1]:
import asyncio
import json
import time as _time
import xml.etree.ElementTree as ET

import anywidget
import traitlets

ET.register_namespace("", "http://www.w3.org/2000/svg")


### The widget

`FormalVizWidget` renders `_svg_content` into the DOM once, then applies
`widget[key] = (fn, method)` bindings to the live DOM every time the
traitlets `state` dict changes: Python evaluates each binding's `fn`
against the new `state` and syncs only the resulting values, and the JS
side interpolates between the old and new values using the `tween`/`snap`
helpers.

`state` is just what's currently on screen. Each widget instance also
keeps its own local, Python-only history of how it got there
(`_history`/`_events`/`_cursor`) so that:

- `w.snapshot()` can fork off a brand new, independent widget (its own
  model, its own DOM) from wherever `w` currently is.
- `w.replay()`, `w.export_trace()`, `w.goto_transition(i)`, `w.prev()`,
  `w.next()` can navigate or dump that history without re-solving the
  spec, and without disturbing that history's ability to differ between
  `w` and any widget forked from it.

In [2]:
ESM = r"""
const SVG_NS = "http://www.w3.org/2000/svg";

export default {
  render({ model, el }) {
    const container = document.createElement("div");
    el.appendChild(container);

    let svg = null;
    let customFns = {};      // name -> compiled custom transition fn
    let animEls = new Map(); // binding key -> live <animate>/<animateTransform> node

    // native svg animation (SMIL) instead of css transitions. These
    // elements live inside the svg itself, so python can build the exact
    // same kind of thing from history for a standalone/exported svg
    // (see _rendered_svg / _append_smil_trace).
    function fmtTransformArgs(args) {
      return args.join(" ");
    }

    function fmtTransformAttr([kind, args]) {
      return `${kind}(${args.join(" ")})`;
    }

    // begin="indefinite" elements need an explicit (re)start. calling
    // beginElement() while already active *adds* an overlapping interval
    // instead of replacing it, so always end the current one first.
    function retrigger(smil) {
      try { smil.endElement(); } catch (e) {}
      smil.beginElement();
    }

    // get (or lazily create) the animate/animateTransform node for a
    // binding key. transform bindings need animateTransform, everything
    // else a plain animate.
    function ensureSmil(el, key, meta) {
      const tag = meta.prop === "transform" ? "animateTransform" : "animate";
      let smil = animEls.get(key);
      if (smil && smil.tagName.toLowerCase() !== tag.toLowerCase()) {
        smil.remove();
        smil = null;
      }
      if (!smil) {
        smil = document.createElementNS(SVG_NS, tag);
        smil.setAttribute("attributeName", meta.prop === "transform" ? "transform" : meta.prop);
        smil.setAttribute("begin", "indefinite");
        smil.setAttribute("fill", "freeze");
        smil.setAttribute("repeatCount", "1");
        el.appendChild(smil);
        animEls.set(key, smil);
      }
      return smil;
    }

    // apply one tween/snap binding. value/prev come precomputed from
    // python (fn(state) evaluated there, never in JS). from/to for the
    // animation are always {prev, value} exactly as python computed them,
    // not whatever's currently on screen, so a retrigger mid-flight is
    // deterministic rather than depending on browser frame timing.
    function applyTweenOrSnap(key, meta, entry, duration) {
      const el = resolveEl(meta.selector);
      if (!el) return;
      const { value, prev } = entry;
      const instant = meta.method === "snap" || prev === null || !(duration > 0);

      if (meta.prop === "transform") {
        if (instant || value[0] !== prev[0]) {
          if (!instant) {
            console.warn(
              `widget[${JSON.stringify(key)}]: transform op kind changed ` +
              `${prev[0]} -> ${value[0]}; snapping instead of animating`
            );
          }
          const existing = animEls.get(key);
          if (existing) existing.setAttribute("begin", "indefinite");
          el.setAttribute("transform", fmtTransformAttr(value));
          return;
        }
        const smil = ensureSmil(el, key, meta);
        smil.setAttribute("type", value[0]);
        el.setAttribute("transform", fmtTransformAttr(prev));
        smil.setAttribute("from", fmtTransformArgs(prev[1]));
        smil.setAttribute("to", fmtTransformArgs(value[1]));
        smil.setAttribute("dur", (duration / 1000) + "s");
        retrigger(smil);
        return;
      }

      if (instant) {
        const existing = animEls.get(key);
        if (existing) existing.setAttribute("begin", "indefinite");
        el.setAttribute(meta.prop, value);
        return;
      }
      const smil = ensureSmil(el, key, meta);
      el.setAttribute(meta.prop, prev);
      smil.setAttribute("from", prev);
      smil.setAttribute("to", value);
      smil.setAttribute("dur", (duration / 1000) + "s");
      retrigger(smil);
    }

    // compile widget[key] = (fn, "customName") custom transitions, registered
    // from python via register_transition(name, code). code is a function
    // BODY (statements), signature (el, value, prev, duration).
    function buildCustomFns() {
      customFns = {};
      const table = model.get("_custom_transitions") || {};
      for (const [name, code] of Object.entries(table)) {
        try {
          customFns[name] = new Function("el", "value", "prev", "duration", code);
        } catch (e) {
          console.error(`register_transition(${JSON.stringify(name)}) compile error:`, e);
        }
      }
    }

    function resolveEl(selector) {
      if (!svg) return null;
      return svg.getElementById(selector) || svg.querySelector(selector);
    }

    // apply every widget[key] = (fn, method) binding using its precomputed value.
    // durationOverride forces instant application (inital).
    function applyAllBindings(durationOverride) {
      const meta = model.get("_bindings_meta") || {};
      const values = model.get("_binding_values") || {};
      const duration = durationOverride ?? model.get("_transition_duration");

      for (const key of Object.keys(meta)) {
        const m = meta[key];
        const entry = values[key];
        if (!m || !entry) continue;
        const el = resolveEl(m.selector);
        if (!el) continue;
        const { value, prev } = entry;

        if (m.prop === "text") {
          // pseudo-property: textContent, never animated
          el.textContent = value;
        } else if (m.prop.startsWith("attr:")) {
          // pseudo-property: raw setAttribute, never animated
          el.setAttribute(m.prop.slice("attr:".length), value);
        } else if (m.method === "tween" || m.method === "snap") {
          applyTweenOrSnap(key, m, entry, duration);
        } else {
          const fn = customFns[m.method];
          if (fn) {
            fn(el, value, prev, duration);
          } else {
            console.warn(
              `widget[${JSON.stringify(key)}] uses transition "${m.method}" ` +
              `but no such transition was registered. Call ` +
              `register_transition(${JSON.stringify(m.method)}, ...) first.`
            );
          }
        }
      }
    }

    // render SVG
    function renderSVG() {
      container.innerHTML = model.get("_svg_content");
      svg = container.querySelector("svg");
      animEls = new Map(); // fresh DOM, any prior animate nodes are gone with it
      if (!svg) return;
      // sometimes we want to change viewbox. look at elevator example
      const vb = model.get("_viewbox");
      if (vb) svg.setAttribute("viewBox", vb);
      // apply init state, instantly (duration 0 either way)
      applyAllBindings(0);
    }

    model.on("change:_svg_content", renderSVG);
    model.on("change:_binding_values", () => applyAllBindings());
    model.on("change:_bindings_meta", () => applyAllBindings());
    model.on("change:_custom_transitions", () => { buildCustomFns(); applyAllBindings(); });

    buildCustomFns();
    renderSVG();
  }
};
"""


In [3]:
class FormalVizWidget(anywidget.AnyWidget):
    _esm = ESM

    _svg_content = traitlets.Unicode("").tag(sync=True)
    _viewbox = traitlets.Unicode("").tag(sync=True)
    _max_width = traitlets.Unicode("800px").tag(sync=True)
    _transition_duration = traitlets.Float(300).tag(sync=True)  # ms

    # widget[key] = (fn, method) bindings: static per-key metadata
    # ({selector, prop, method}, keyed by the same "selector.prop" string
    # the user indexed with) and the live per-key {value, prev} computed
    # by evaluating fn(state) in python. see __setitem__.
    _bindings_meta = traitlets.Dict({}).tag(sync=True)
    _binding_values = traitlets.Dict({}).tag(sync=True)
    # name -> JS function body, registered via register_transition() and
    # usable as the method in widget[key] = (fn, name)
    _custom_transitions = traitlets.Dict({}).tag(sync=True)

    # the displayed state, mirrored to JS. always equal to
    # self._history[self._cursor]
    state = traitlets.Dict({}).tag(sync=True)

    # set by a subclass to a Spectabular relation spec: a `VectorTable`, or
    # a `Table` selecting among named relations, or
    # several combined with operators.
    # basically what people would expect
    spec = None

    def __init__(self, svg_content="", variables=None,
                 viewbox=None, max_width="800px", **kw):
        initial = dict(variables or {})
        super().__init__(
            _svg_content=svg_content,
            state=dict(initial),
            _viewbox=viewbox or "",
            _max_width=max_width,
            **kw, # allow user to edit anywidget settings
        )
        self._initial = initial
        # transition history: _history[i] is the state after _events[i-1]
        # fired (_history[0] is always the initial state, so
        # len(_history) == len(_events) + 1). _cursor indexes into
        # _history for whichever point is currently on screen, normally
        # the most recent entry, but goto_transition/prev/next can move it
        # back without discarding anything, so scrubbing through history
        # doesn't require re-running the formal model.
        self._history = [dict(initial)]
        self._events = []
        # seconds per recorded transition; self._durations[i] is how long
        # the step _history[i] -> _history[i+1] took, so
        # len(_durations) == len(_events) == len(_history) - 1. Only
        # needed to lay out keyTimes for the exported SMIL trace (see
        # _append_smil_trace). The live widget gets duration from
        # _transition_duration instead.
        self._durations = []
        self._cursor = 0
        # key ("selector.prop") -> (selector, prop, fn, method). python-only,
        # can't be a traitlet since it holds callables; _bindings_meta/
        # _binding_values are the synced projections of this.
        self._bindings = {}

    # if the widget is not supported in export, fallback to svg. _svg_content
    # itself never changes after construction. The JS side renders it into
    # the DOM once, then mutates that live DOM directly as state/bindings
    # change (see ESM's renderSVG/applyAllBindings), never writing back to
    # the traitlet. So a plain `self._svg_content` here would always be the
    # widget's *initial* frame, regardless of what's actually on screen (see
    # e.g. Elevator.ipynb's per-step snapshot figures). Bake the current
    # widget[key] = (fn, method) binding values into a copy instead:
    # every visible change goes through a binding (there's no other way to
    # touch the DOM anymore), so this is exhaustive, not a best effort.
    def _repr_mimebundle_(self, **kwargs):
        data, metadata = super()._repr_mimebundle_(**kwargs)

        data["image/svg+xml"] = self._rendered_svg()

        return data, metadata

    def _rendered_svg(self):
        if not self._bindings or not self._svg_content:
            return self._svg_content
        try:
            root = ET.fromstring(self._svg_content)
        except ET.ParseError:
            return self._svg_content
        by_id = {el.get("id"): el for el in root.iter() if el.get("id")}

        # trace up to whatever's currently displayed (goto_transition/prev
        # can leave the cursor short of the latest recorded step). Export
        # follows the screen, not the full history.
        history = self._history[: self._cursor + 1]
        durations = self._durations[: self._cursor]
        total_dur = sum(durations)

        for key, (selector, prop, fn, method) in self._bindings.items():
            el = by_id.get(selector)
            if el is None:
                continue
            entry = self._binding_values.get(key)
            if entry is None:
                continue
            final_value = entry["value"]

            # base attribute always ends up as the *final* frame, whether
            # or not anything below can animate it. This is what a
            # SMIL-blind consumer (e.g. compile.sh's pdf export) falls
            # back to, so it must never be stuck on the first frame.
            if prop == "text":
                el.text = str(final_value)
                continue  # never animated: SMIL text-content isn't reliable cross-browser
            elif prop.startswith("attr:"):
                el.set(prop.removeprefix("attr:"), str(final_value))
                continue  # never animated, same as "text"
            elif prop == "transform":
                el.set("transform", _transform_attr_str(final_value))
            else:
                # a plain SVG presentation attribute (fill, opacity, ...),
                # matching how the live widget sets it (see ESM's
                # applyTweenOrSnap): setAttribute, not style, so the
                # <animate attributeName="..."> appended below (which
                # targets the same attribute) isn't masked by an inline
                # style declaration for the same property.
                el.set(prop, str(final_value))

            if method not in ("tween", "snap"):
                continue  # custom transition: arbitrary JS, no SMIL equivalent, base bake above is final
            if total_dur <= 0 or len(history) < 2:
                continue  # nothing recorded to animate over

            values_seq = [fn(h) for h in history]
            self._append_smil_trace(el, prop, method, values_seq, durations, total_dur)

        return ET.tostring(root, encoding="unicode")

    @staticmethod
    def _append_smil_trace(el, prop, method, values_seq, durations, total_dur):
        # one <animate>/<animateTransform> spanning the whole displayed
        # trace, values/keyTimes keyed by frame. Same values/keyTimes
        # idiom as drawsvg/animator.py, built with ET.SubElement instead
        # of string injection since _rendered_svg already parses the SVG.
        cum, acc = [0.0], 0.0
        for d in durations:
            acc += d
            cum.append(acc)
        key_times = [c / total_dur for c in cum]
        key_times[-1] = 1.0  # guard float drift, SMIL requires exactly 1.0
        key_times_str = ";".join(_fmt_num(kt) for kt in key_times)
        calc_mode = "discrete" if method == "snap" else "linear"

        if prop == "transform":
            kinds = {op[0] for op in values_seq}
            if len(kinds) != 1:
                raise ValueError(
                    f"{el.get('id')!r}.transform binding returned mixed op kinds "
                    f"{sorted(kinds)} across its history; a .transform binding's "
                    "fn must always return the same kind"
                )
            anim = ET.SubElement(el, "animateTransform")
            anim.set("attributeName", "transform")
            anim.set("type", kinds.pop())
            anim.set("values", ";".join(_transform_args_str(op[1]) for op in values_seq))
        else:
            anim = ET.SubElement(el, "animate")
            anim.set("attributeName", prop)
            anim.set("values", ";".join(str(v) for v in values_seq))

        anim.set("keyTimes", key_times_str)
        anim.set("dur", f"{_fmt_num(total_dur)}s")
        anim.set("calcMode", calc_mode)
        anim.set("repeatCount", "1")
        anim.set("fill", "freeze")
        anim.set("begin", "0s")

    # make this from svg
    @classmethod
    def from_svg(cls, path, variables, viewbox=None, **kw):
        with open(path) as f:
            svg = f.read()
        return cls(svg, variables, viewbox, **kw)

    @property
    def s(self):
        return _S(self) #delegate

    # --- declarative SVG bindings -------------------------------------
    #
    # widget["selector.prop"] = (fn, method)
    #   fn      : state -> value. called with the widget's current state
    #             whenever state changes (and immediately on assignment).
    #   method  : "tween" (animate via a native SVG <animate>/
    #             <animateTransform> element), "snap" (apply instantly),
    #             or the name of a transition registered with
    #             register_transition() (called as fn(el, value, prev,
    #             duration) on the JS side).
    #   prop    : "text" sets el.textContent instead of an attribute;
    #             "attr:name" sets the raw SVG attribute "name" via
    #             setAttribute instead. Both are applied instantly
    #             regardless of method, since neither has a SMIL
    #             animation equivalent (SMIL only animates presentation
    #             attributes, not text content). "transform" is special:
    #             fn must return a structured op `(kind, args)` instead of
    #             a plain value, since <animateTransform> needs a fixed
    #             `type` plus bare numeric args rather than a CSS
    #             transform string, e.g. `("translate", (x, y))`,
    #             `("rotate", (deg, cx, cy))`, or `("scale", (sx, sy))`. A
    #             given .transform binding must return the same `kind`
    #             for every state it's evaluated on. Every other prop is
    #             a plain SVG presentation attribute (`fill`, `opacity`,
    #             ...), set directly via setAttribute.
    #
    # javascript never sees fn. python evaluates it and syncs only the
    # resulting value, same as every other state change.

    def __setitem__(self, key, spec):
        try:
            fn, method = spec
        except (TypeError, ValueError):
            raise TypeError(
                f"widget[{key!r}] = (fn, method): fn(state) -> value, "
                "method is 'tween', 'snap', or a name registered with "
                "register_transition()"
            ) from None
        if not callable(fn):
            raise TypeError(f"binding value for {key!r} must be callable(state) -> value")
        if not isinstance(method, str):
            raise TypeError(f"binding method for {key!r} must be a string")
        selector, prop = _parse_binding_key(key)
        self._bindings[key] = (selector, prop, fn, method)
        self._bindings_meta = {
            **self._bindings_meta,
            key: {"selector": selector, "prop": prop, "method": method},
        }
        self._recompute_binding(key)

    def __getitem__(self, key):
        # (fn, method), as assigned
        return self._bindings[key][2:]

    def __delitem__(self, key):
        del self._bindings[key]
        meta = dict(self._bindings_meta)
        meta.pop(key, None)
        self._bindings_meta = meta
        values = dict(self._binding_values)
        values.pop(key, None)
        self._binding_values = values

    def __contains__(self, key):
        return key in self._bindings

    def register_transition(self, name, code):
        """
        Register a custom JS transition under `name`, usable as the
        method in `widget[key] = (fn, name)`. `code` is a JS function
        BODY (statements, not an arrow expression), compiled with
        parameters `(el, value, prev, duration)`:

            w.register_transition("pulse", '''
                el.animate([{ opacity: 0.3 }, { opacity: 1 }], duration);
            ''')
            w["alarm.fill"] = (lambda s: "#f00" if s["alarm"] else "#333", "pulse")
        """
        if name in ("tween", "snap"):
            raise ValueError(f"{name!r} is a built-in transition method")
        self._custom_transitions = {**self._custom_transitions, name: code}

    def _recompute_binding(self, key):
        selector, prop, fn, method = self._bindings[key]
        value = fn(self.state)
        if prop == "transform":
            _validate_transform_op(key, value)
        self._binding_values = {
            **self._binding_values,
            key: {"value": value, "prev": None},
        }

    def _recompute_all_bindings(self, state, prev_state):
        if not self._bindings:
            return
        values = dict(self._binding_values)
        for key, (selector, prop, fn, method) in self._bindings.items():
            value = fn(state)
            if prop == "transform":
                _validate_transform_op(key, value)
            values[key] = {"value": value, "prev": fn(prev_state)}
        self._binding_values = values

    # --------------------------------------------------------------------

    # sets what's on screen without using history. used both by the
    # transition/snap methods below (which do record) and by history
    # navigation (which deliberately doesn't; moving around in time you
    # already have isn't a new transition).

    def _display(self, state, duration):
        prev_state = self.state
        self._transition_duration = duration * 1000  # ms for JS
        self.state = dict(state)
        self._recompute_all_bindings(self.state, prev_state)

    def _record(self, next_state, event, label, duration):
        # if the cursor isn't at the head (the user rewound with prev()/
        # goto_transition() and is now doing something new)
        del self._history[self._cursor + 1:]
        del self._events[self._cursor:]
        del self._durations[self._cursor:]
        self._events.append(event if event is not None else label)
        self._history.append(dict(next_state))
        self._durations.append(duration)
        self._cursor = len(self._history) - 1

    def reset(self):
        # back to the initial state, and forget the history that led
        # anywhere else
        self._history = [dict(self._initial)]
        self._events = []
        self._durations = []
        self._cursor = 0
        self._display(self._initial, 0)

    async def transition(self, next_state, duration=0.3, event=None, label=None):
        # await until transition
        self._record(next_state, event, label, duration)
        self._display(next_state, duration)
        await asyncio.sleep(duration)

    def transition_sync(self, next_state, duration=0.3, event=None, label=None):
        # block until transition
        self._record(next_state, event, label, duration)
        self._display(next_state, duration)
        _time.sleep(duration)

    def snap(self, next_state, event=None, label=None):
        # snap
        self._record(next_state, event, label, 0.0)
        self._display(next_state, 0)

    # move the display around in already-recorded history. non-animated
    # by default since these are typically single scrub steps run from
    # their own cell; pass duration for a tween instead.

    def goto_transition(self, index, duration=0.0):
        index = max(0, min(index, len(self._history) - 1))
        self._cursor = index
        self._display(self._history[index], duration)

    def prev(self, duration=0.0):
        if self._cursor > 0:
            self.goto_transition(self._cursor - 1, duration)

    def next(self, duration=0.0):
        if self._cursor < len(self._history) - 1:
            self.goto_transition(self._cursor + 1, duration)

    @property
    def cursor(self):
        return self._cursor

    @property
    def can_prev(self):
        return self._cursor > 0

    @property
    def can_next(self):
        return self._cursor < len(self._history) - 1

    @property
    def history(self):
        # states visited, index 0 is always the initial state
        return [dict(s) for s in self._history]

    @property
    def durations(self):
        # seconds per recorded transition, durations[i] is history[i] -> history[i+1]
        return list(self._durations)

    def __len__(self):
        # number of recorded transitions (== len(history) - 1)
        return len(self._events)

    async def replay(self, tick_dt=0.5, transition_dur=0.15, from_index=0, to_index=None):
        """
        Re-animate through already-recorded history, in order,
        without recomputing states
        """
        if to_index is None:
            to_index = len(self._history) - 1
        self.goto_transition(from_index, duration=0)
        for i in range(from_index + 1, to_index + 1):
            self.goto_transition(i, duration=transition_dur)
            await asyncio.sleep(transition_dur)
            await asyncio.sleep(max(0.0, tick_dt - transition_dur))

    def export_trace(self, as_json=False):
        trace = [
            {
                "index": i,
                "event": self._events[i],
                "before": self._history[i],
                "after": self._history[i + 1],
                "duration": self._durations[i],
            }
            for i in range(len(self._events))
        ]
        return json.dumps(trace, default=str) if as_json else trace

    def snapshot(self):
        """
        A new, fully independent widget (its own model and DOM) showing
        this widget's current displayed state. Carries along the history up
        to the cursor.
        """
        cls = type(self)
        copy = cls(
            svg_content=self._svg_content,
            variables=self._history[self._cursor],
            viewbox=self._viewbox,
            max_width=self._max_width,
        )
        copy._history = [dict(s) for s in self._history[:self._cursor + 1]]
        copy._events = list(self._events[:self._cursor])
        copy._durations = list(self._durations[:self._cursor])
        copy._cursor = self._cursor
        copy._bindings = dict(self._bindings)
        copy._bindings_meta = dict(self._bindings_meta)
        copy._custom_transitions = dict(self._custom_transitions)
        copy._recompute_all_bindings(copy.state, copy.state)
        return copy

    @classmethod
    def _flattened_spec(cls):
        """
        `cls.spec`, flattened and indexed by free variable name, cached on
        the concrete subclass.
        had to use cls.__dict__ instead of hasattr to prevent shadowing from parent
        """
        if "_flat_spec_cache" not in cls.__dict__:
            if cls.spec is None:
                raise NotImplementedError(f"{cls.__name__}.spec is not set")
            flat = flatten(cls.spec)
            free_by_name = {str(v): v for v in freevars(flat)}
            if isinstance(cls.spec, VectorTable):
                # The variables the pecs assigns values to, one per row of
                # .left. primed (a relation table's next state values) or
                # unprimed (a predicate table's values).
                output_names = {str(v) for v in cls.spec.left}
            else:
                # No `.left` to read (like a Table()).
                # a free variable ending in `ʹ` is a next-state output, everything
                # else a read-only input.
                output_names = {name for name in free_by_name if name.endswith("ʹ")}
            cls._flat_spec_cache = (flat, free_by_name, output_names)
        return cls._flat_spec_cache

    @classmethod
    def compute_next_state(cls, state, **event):
        """
        Solve `cls.spec` for the values of whatever it assigns to
        given `state` plus whatever else a particular row needs.

        Works uniformly for both roles a VectorTable can play:
        - a relation table, where `.left` holds primed (`xʹ`) next-state
          variables. This is the usual case,the result is a next state,
          keyed by the unprimed names.
        - a **predicate table**, where `.left` holds unprimed
          variables derived from the current state. Itreturns
          those derived values, keyed by their own names.

        Every other free variable in the spec is a read only input.
        We pin from `state` if present there, else from `event` if supplied.
        An unprimed variable that's the counterpart of a primed output (i.e.
        clearly meant to be persisted state) must be in `state`, or this
        raises `KeyError`. Any other input (an event selector, a
        predicate's own extra parameter, ...) is simply left unconstrained
        if missing. Harmless if the row that fires doesn't actually
        depend on it.

        A classmethod rather than an instance method (and `spec` a class
        variable rather than instance state) so it also works without a
        live widget.
        """
        flat, free_by_name, output_names = cls._flattened_spec()
        #print("flat:", flat)
        #print("free: ", free_by_name)
        #print("output_names:", output_names)
        input_names = set(free_by_name) - output_names
        required_state_names = {name.removesuffix("ʹ") for name in output_names if name.endswith("ʹ")}
        #print(required_state_names)

        solver = z3.Solver()
        solver.add(flat)
        for name in input_names:
            if name in state:
                # read any free vars from state
                value = state[name]
            elif name in event:
                # read any free vars from event
                value = event[name]
            elif name in required_state_names:
                raise KeyError(f"state is missing {name!r}, required by {cls.__name__}.spec")
            else:
                continue  # not relevant to whichever row fires; let Z3 pick freely
            var = free_by_name[name]
            solver.add(var == _state_value_to_z3(value, var.sort()))

        result = solver.check()
        if result != z3.sat:
            raise ValueError(
                f"{cls.__name__}.spec has no enabled row for {event!r} from state {state!r} ({result})"
            )
        # a table can have several satisfying assignments for its
        # output variables (nondeterminism). this returns whichever one
        # Z3 happens to find, which is fine for scripted/replayed
        # animation (the scenario already picked the transition/row via
        # `event`) but not for interactively exploring all enabled next
        # states, which would need re-solving under a blocking clause per
        # answer found.
        model = solver.model()
        return {
            name.removesuffix("ʹ"): _z3_to_state_value(model.eval(free_by_name[name], model_completion=True))
            for name in output_names
        }


def _parse_binding_key(key):
    # "selector.prop" -> ("selector", "prop"); selector is looked up via
    # getElementById first, then as a CSS selector, so plain element ids
    # (the common case) and full selectors both work as long as the
    # selector itself doesn't contain a dot.
    if not isinstance(key, str) or "." not in key:
        raise ValueError(
            f"binding key {key!r} must look like 'selector.prop', e.g. "
            "'button1up.fill' or 'floor-display.text'"
        )
    selector, prop = key.rsplit(".", 1)
    if not selector or not prop:
        raise ValueError(f"binding key {key!r} must look like 'selector.prop'")
    return selector, prop


# .transform bindings return (kind, args) instead of a plain value. See
# the "declarative SVG bindings" comment above __setitem__. kind fixes
# which <animateTransform type="..."> to use, arity is just a sanity check
# (animateTransform can't animate a mismatched-arity op anyway).
_TRANSFORM_ARITY = {"translate": 2, "rotate": 3, "scale": 2}


def _validate_transform_op(key, op):
    if not (isinstance(op, tuple) and len(op) == 2 and op[0] in _TRANSFORM_ARITY):
        raise TypeError(
            f"widget[{key!r}] (.transform) must return (kind, args), kind "
            f"one of {sorted(_TRANSFORM_ARITY)}; got {op!r}"
        )
    kind, args = op
    arity = _TRANSFORM_ARITY[kind]
    if not (isinstance(args, tuple) and len(args) == arity
            and all(isinstance(a, (int, float)) for a in args)):
        raise TypeError(
            f"widget[{key!r}] transform op {kind!r} needs a {arity}-tuple "
            f"of numbers, got {args!r}"
        )


def _fmt_num(x):
    # avoid scientific notation, SMIL's number/clock-value grammar doesn't accept it
    if isinstance(x, float) and x.is_integer():
        return str(int(x))
    return f"{x:.6f}".rstrip("0").rstrip(".")


def _transform_args_str(args):
    return " ".join(_fmt_num(a) for a in args)


def _transform_attr_str(op):
    # (kind, args) -> the SVG1.1 `transform` attribute string, e.g.
    # ("translate", (0, 40)) -> "translate(0 40)"
    kind, args = op
    return f"{kind}({_transform_args_str(args)})"


class _S:
    # attribute access of a widget's state (w.s.blinkRight instead of w.state["blinkRight"]).

    def __init__(self, widget):
        object.__setattr__(self, "_widget", widget)

    def __getattr__(self, name):
        try:
            return self._widget.state[name]
        except KeyError:
            raise AttributeError(name) from None

    def __setattr__(self, name, value):
        # allows us to use the ** syntax. goes through snap() (not the
        # trait directly) so direct field edits still land in history.
        next_state = dict(self._widget.state)
        next_state[name] = value
        self._widget.snap(next_state, event={name: value})

    def __repr__(self):
        return repr(self._widget.state)


def _state_value_to_z3(value, sort):
    if isinstance(value, bool):
        return z3.BoolVal(value)
    if isinstance(value, int):
        return z3.IntVal(value)
    if isinstance(value, float):
        return z3.RealVal(value)
    if isinstance(value, str):
        for i in range(sort.num_constructors()):
            if sort.constructor(i).name() == value:
                return sort.constructor(i)()
        raise ValueError(f"{value!r} is not a value of enum sort {sort}")
    raise TypeError(f"don't know how to convert {value!r} (a {type(value).__name__}) to a Z3 term")


def _z3_to_state_value(value):
    kind = value.sort().kind()
    if kind == z3.Z3_BOOL_SORT:
        return z3.is_true(value)
    if kind == z3.Z3_INT_SORT:
        return value.as_long()
    if kind == z3.Z3_REAL_SORT:
        return value.as_fraction()
    return str(value)  # enum (or other datatype) constructor name


TODO:
- Expose multiple satisfying next states for a nondeterministic relation
  (re-solve per model found) to support
  interactive exploration rather than only scripted replay.
- Catch trying to use 2D Tables in the animation. Maybe we put this onus on the animator?

### Scenario runners
Helpers to run scenarios.

In [4]:
async def run_scenario(w, events, tick_dt=0.5, transition_dur=0.15):
    """
    Run a sequence of events, animating smoothly between each.

    w : FormalVizWidget
    events : list of dicts, one set of `compute_next_state` keyword arguments per event
    tick_dt : seconds to wait after each event
    transition_dur : animation duration per transition
    """
    for event in events:
        next_state = w.compute_next_state(w.state, **event)
        await w.transition(next_state, duration=transition_dur, event=event)
        await asyncio.sleep(max(0.0, tick_dt - transition_dur))


async def advance_ticks(w, n, event, tick_dt=0.5, transition_dur=0.15, stop_when=None):
    """
    Send the same event `n` times in a row animating between each.
    Stops early if `stop_when(state)` becomes true. Usefull for
    modeling time based applications

    Returns the number of events actually sent.
    """
    for i in range(n):
        if stop_when is not None and stop_when(w.state):
            return i
        next_state = w.compute_next_state(w.state, **event)
        await w.transition(next_state, duration=transition_dur, event=event)
        await asyncio.sleep(max(0.0, tick_dt - transition_dur))
    return n


### General helpers
useful for debugging or testing any `compute_next_state` function against any widget.

In [5]:
def diff_state(before, next):
    # return what varibles in state change from before to next
    keys = before.keys() | next.keys()
    return {k: (before.get(k), next.get(k)) for k in keys if before.get(k) != next.get(k)}


def trace_scenario(widget_cls, initial_state, events):
    # tracer with no animation
    state = dict(initial_state)
    trace = []
    for event in events:
        next_state = widget_cls.compute_next_state(state, **event)
        trace.append((event, dict(state), dict(next_state)))
        state = next_state
    return trace


async def reset_and_run(w, scenario_fn, *args, settle=0.3, **kwargs):
    """
    Reset a widget to its initial state, let the reset render settle, then
    run `scenario_fn(w, *args, **kwargs)` (e.g. a demo_* function or
    `run_scenario`). 
    """
    w.reset()
    await asyncio.sleep(settle)
    await scenario_fn(w, *args, **kwargs)

### Using this framework

This notebook defines `FormalVizWidget` (with its generic
`compute_next_state`), `run_scenario`, `advance_ticks`, `diff_state`,
`trace_scenario`, and `reset_and_run` in the calling kernel once `%run`
has executed it. A model notebook:

1. `%run`s this notebook, then `%run`s `spectabular.ipynb` for
   `Bool`/`Int`/`EnumType`/`VectorTable`/`flatten`/...
2. builds its spec and subclasses `FormalVizWidget` with `spec = ...` set
   as a class variable.

See `elevator/Elevator.ipynb` for a full worked example.

#### Changing state

- `w.transition(next_state, duration=0.3, event=None, label=None)`.
  async, animates and records a new step in `w`'s history.
- `w.transition_sync(...)`. same, but blocks instead of `await`ing.
- `w.snap(next_state, event=None, label=None)`. instant, still recorded.
- `w.s.name = value`. shorthand for a one-field `snap`; also recorded.
- `w.reset()`. back to the initial state, discarding all history.

`event`/`label` is whatever you want attached to that step for
`export_trace()` later, typically the same kwargs dict passed to
`compute_next_state`, which is what `run_scenario`/`advance_ticks` record
automatically.

#### Forking: `snapshot()`

```python
w1 = w.snapshot()   # independent widget, currently identical to w
w.transition(...)   # only w moves
w1.transition(...)  # only w1 moves, from where it was snapshotted
```

`snapshot()` returns a new widget of the same class, showing wherever
`w`'s cursor currently is (not necessarily its latest state, see
below), carrying along the history that led there. From that point on
the two widgets never affect each other.

#### Transition history

- `w.history` - list of every state visited, `history[0]` is the
  initial state.
- `w.durations` - seconds each recorded transition took,
  `durations[i]` is `history[i] -> history[i+1]`.
- `w.cursor`, `len(w)` - current position, and number of recorded
  transitions.
- `w.export_trace(as_json=False)` - list (or JSON string) of
  `{index, event, before, after, duration}` per transition.
- `w.goto_transition(i, duration=0.0)` - jump the display to
  `history[i]` without re-solving anything or changing what's recorded.
- `w.prev(duration=0.0)` / `w.next(duration=0.0)` - step the cursor by
  one; `w.can_prev` / `w.can_next` say whether there's anywhere to go.
- `await w.replay(tick_dt=0.5, transition_dur=0.15, from_index=0, to_index=None)`
  - re-animate through recorded history in order. Unlike re-running
  `run_scenario`, this never calls `compute_next_state` again, so it
  reproduces exactly what happened even if the spec is nondeterministic.

Transitioning after rewinding with `prev()`/`goto_transition()` truncates
whatever history was ahead of the cursor before recording new stuff. `w.snapshot()` can
preserve a branch point instead of overwriting it.

#### Binding SVG properties to state

For the common case, where some property of some element should just track
a function of `state`, you don't have to hand-write JS at all:

```python
w["cabin-group.transform"] = (
    lambda s: ("translate", (0, FLOOR_Y[s["floor"]] - FLOOR_Y["F1"])),
    "tween",
)
w["btn-b1u.fill"] = (lambda s: "#ff9800" if s["b1u"] else "#bdbdbd", "tween")
w["floor-display.text"] = (lambda s: s["floor"], "snap")
```

`widget[key] = (fn, method)`:

- `key` is `"selector.prop"`. `selector` is looked up with
  `getElementById` first, then as a CSS selector, so a plain element id
  (the usual case) and a full selector both work, as long as the
  selector itself doesn't contain a dot. `prop` is normally a plain SVG
  presentation attribute (`fill`, `opacity`, ...), set via
  `setAttribute` so it can be driven by a native `<animate>` element (see
  `method` below). `"transform"` is special-cased: `fn` must return a
  structured op `(kind, args)` instead of a plain value, e.g.
  `("translate", (x, y))`, `("rotate", (deg, cx, cy))`, or
  `("scale", (sx, sy))`, since `<animateTransform>` needs a fixed
  `type` plus bare numeric args, not a CSS transform string. A given
  `.transform` binding must return the same `kind` on every state it's
  evaluated on. Two further pseudo-properties are handled specially and
  always apply instantly (no `.transform`-style structure, no
  animation): `"text"` sets `el.textContent`, and `"attr:name"` sets the
  raw SVG attribute `name` via `setAttribute` (for things not otherwise
  reachable, like `d`).
- `fn` is `state -> value`, called in **Python** every time `state`
  changes (and once immediately on assignment). JS never sees `fn`.
  Only the resulting value is synced, so ordinary Python (closures,
  helper functions, whatever) is fine inside it.
- `method` is `"tween"` (animate via a native SVG `<animate>`/
  `<animateTransform>` element running from the previous value to the
  new one over `_transition_duration`), `"snap"` (apply instantly), or
  the name of a transition registered with `register_transition()`
  below.

`del widget[key]` removes a binding; `key in widget` checks for one;
`widget[key]` returns back the `(fn, method)` you assigned.
`w.snapshot()` carries all of a widget's bindings over to the fork.

Bindings compose with everything above for free: they're recomputed
inside `_display()`, so `transition()`/`snap()`/`goto_transition()`/
`prev()`/`next()`/`replay()` all drive them the same way they drive
`state`.

If several bindings on the *same element* change together (e.g. an
indicator light's `fill`, `fill-opacity`, and `stroke-opacity` all
tweening on together), give each its own `widget[key] = (fn, "tween")`
entry: each gets its own independent `<animate>` element, and since
they all start at the same state change they play in sync without
needing to be grouped into one animation.

Because every visible change is a `widget[key] = (fn, method)` entry
computed in Python, `_repr_mimebundle_` (see `_rendered_svg()`) doesn't
just bake in whatever's on screen right now: it embeds the *whole*
displayed trace (`w.history[0]` through the cursor) as native SVG
animation elements. A static export (PDF, a non-JS HTML render, opening
the SVG file directly) replays the recorded scenario on its own, no
Jupyter kernel or JS widget required.

##### Custom transitions

`"tween"`/`"snap"` cover most cases, but sometimes you want real control
over how one property animates (multi-step, needs to compare against the
previous value, etc). Register a named JS transition once, then use its
name as the method for any binding:

```python
w.register_transition("pulse", '''
    el.animate([{ opacity: 0.2 }, { opacity: 1 }], duration);
''')
w["alarm.fill"] = (lambda s: "#f00" if s["alarm"] else "#333", "pulse")
```

`code` is a JS function **body** (statements, not an arrow expression),
compiled with the signature `(el, value, prev, duration)`: `el` is the
resolved DOM element, `value`/`prev` are `fn(state)` evaluated on the new
and previous state (`prev` is `None`/`null` the first time a binding is
assigned), and `duration` is the current `_transition_duration` in ms.

`el`, `value`/`prev`, and `duration` are exactly what a built-in `tween`
gets internally. A custom transition is just a way to replace the "set
an attribute" step with arbitrary DOM code for one binding, while still
going through the same `fn(state) -> value` machinery as every other
binding. There's deliberately no broader escape hatch that looks at the
whole state transition at once outside a binding: every visible change on
the widget is a `widget[key] = (fn, method)` entry, computed in Python,
which is what lets `_repr_mimebundle_` reconstruct "what's on screen
right now" (and, for `"tween"`/`"snap"` bindings, how it got there) as a
static SVG without needing a browser. A custom transition has no SMIL
equivalent, though, so it only ever bakes to its final value in that
export. There's no way to replay arbitrary JS outside a live widget.
Something that genuinely needs to touch several elements from one shared
calculation (e.g. reordering elements) is still possible, just as several
bindings sharing the same helper function in their closure rather than as
free-form JS with no Python-side representation.
